In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS houseprices.silver;

## Identificação de nulos

In [0]:
from pyspark.sql.functions import col, count, when, lit, round as spark_round
from pyspark.sql import DataFrame

# Load the bronze table
df_bronze = spark.table("houseprices.bronze.bronze_train")
total_rows = df_bronze.count()

# Analyze missing values (both NULL and "NA" string)
missing_analysis = []

for column_name in df_bronze.columns:
    # Count nulls
    null_count = df_bronze.filter(col(column_name).isNull()).count()
    
    # Count "NA" strings (cast to string to check)
    na_string_count = df_bronze.filter(col(column_name).cast("string") == "NA").count()
    
    # Total missing
    total_missing = null_count + na_string_count
    missing_pct = (total_missing / total_rows) * 100
    
    missing_analysis.append({
        "column_name": column_name,
        "null_count": null_count,
        "na_string_count": na_string_count,
        "total_missing": total_missing,
        "missing_percentage": round(missing_pct, 2),
        "total_rows": total_rows
    })

# Create DataFrame and sort by missing percentage
missing_df = spark.createDataFrame(missing_analysis)
missing_df_sorted = missing_df.orderBy(col("missing_percentage").desc())

display(missing_df_sorted)

In [0]:
df_bronze = spark.table("houseprices.bronze.bronze_train")

schema_fields = sorted(df_bronze.schema.fields, key=lambda f: f.name.lower())
for field in schema_fields:
    print(f"{field.name}: {field.dataType.simpleString()}")

##  Correção do tipo de dados

In [0]:
from pyspark.sql.functions import col

# 1. Carrega a tabela Bronze
df_silver = spark.table("houseprices.bronze.bronze_train")

# 2. Substitui TODOS os textos "NA" por Null verdadeiro de uma vez só
# Como você decidiu tratar tudo como nulo, o .replace() funciona perfeitamente aqui
df_silver = df_silver.replace("NA", None)

# 3. Corrige o tipo das 3 colunas que vieram como texto
df_silver = df_silver.withColumn("LotFrontage", col("LotFrontage").cast("double")) \
                     .withColumn("MasVnrArea", col("MasVnrArea").cast("double")) \
                     .withColumn("GarageYrBlt", col("GarageYrBlt").cast("int"))

# 4. Verifica se a correção funcionou
df_silver.select("LotFrontage", "MasVnrArea", "GarageYrBlt").printSchema()

In [0]:
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import DoubleType, IntegerType, LongType, FloatType, StringType
from pyspark.ml.feature import Imputer

# ==========================================
# 1. SETUP: CARGA E TIPAGEM
# ==========================================
df = spark.table("houseprices.bronze.bronze_train")

# Substitui o texto "NA" por Null globalmente e corrige as colunas mascaradas
df = df.replace("NA", None)
df = df.withColumn("LotFrontage", col("LotFrontage").cast("double")) \
       .withColumn("MasVnrArea", col("MasVnrArea").cast("double")) \
       .withColumn("GarageYrBlt", col("GarageYrBlt").cast("int"))

total_linhas = df.count()
colunas_originais = df.columns

# ==========================================
# 2. DROP DE COLUNAS (> 45% de Nulos)
# ==========================================
# Calcula a contagem de nulos originais para usar como base de decisão
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df.columns
]).first().asDict()

# Remove colunas com mais de 45% de nulos
colunas_para_dropar = [c for c, nulos in null_counts.items() if (nulos / total_linhas) > 0.45]
df = df.drop(*colunas_para_dropar)

# Exibe colunas removidas
print(f"Colunas removidas (>45% nulos): {colunas_para_dropar}")

# ==========================================
# 3. DROP DE LINHAS (> 70% de Nulos)
# ==========================================
linhas_antes_drop = df.count()
limite_minimo_validos = int(len(df.columns) * 0.30)
df = df.dropna(thresh=limite_minimo_validos)
linhas_depois_drop = df.count()
linhas_removidas = linhas_antes_drop - linhas_depois_drop

print(f"Linhas removidas (>70% nulos): {linhas_removidas}")

# ==========================================
# 4. IMPUTAÇÃO DE NUMÉRICOS (Mediana)
# ==========================================
colunas_numericas = [f.name for f in df.schema.fields if isinstance(f.dataType, (DoubleType, IntegerType, LongType, FloatType))]

imputer = Imputer(inputCols=colunas_numericas, outputCols=colunas_numericas).setStrategy("median")
df = imputer.fit(df).transform(df)

# ==========================================
# 5. IMPUTAÇÃO DE CATEGÓRICOS (Híbrida: Moda ou "NA")
# ==========================================
colunas_categoricas = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

dicionario_preenchimento = {}
preenchimento_info = []

for c in colunas_categoricas:
    nulos_col = null_counts.get(c, 0)
    perc_nulos = nulos_col / total_linhas
    
    if perc_nulos > 0.05:
        dicionario_preenchimento[c] = "NA"
        # ADICIONADO "valor_moda": None para manter o padrão das colunas
        preenchimento_info.append({"coluna": c, "estrategia": "NA", "valor_moda": None})
        
    elif nulos_col > 0:
        moda_row = df.groupBy(c).count().filter(col(c).isNotNull()).orderBy(col("count").desc()).first()
        if moda_row:
            dicionario_preenchimento[c] = moda_row[c]
            preenchimento_info.append({"coluna": c, "estrategia": "Moda", "valor_moda": moda_row[c]})

# Aplica todas as regras de preenchimento (tanto as modas quanto os "NA"s) de uma só vez
if dicionario_preenchimento: # Prevenção de erro caso o dicionário esteja vazio
    df = df.fillna(dicionario_preenchimento)

# Visualização das colunas e estratégias de imputação
if preenchimento_info: # Prevenção de erro caso não tenha havido imputação
    preenchimento_df = spark.createDataFrame(preenchimento_info)
    display(preenchimento_df)

# ==========================================
# 6. VALIDAÇÃO FINAL E SALVAMENTO
# ==========================================
df.write.format("delta").mode("overwrite").saveAsTable("houseprices.silver.silver_train")

nulos_restantes = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).first().asDict()
print(f"Total de nulos no dataset inteiro após a limpeza: {sum(nulos_restantes.values())}")

display(df)